# Tokenization and Embeddings

Tokenization is the first step in any NLP pipeline. It converts raw text into a sequence of discrete units (tokens) that a model can process. This notebook covers:

1. **Character-level tokenization** - each character is a token
2. **Word-level tokenization** - each word is a token
3. **Byte-Pair Encoding (BPE)** - subword tokenization concept
4. **Vocabulary building** and token-to-ID mapping
5. **Padding** sequences to uniform length
6. **Visualization** of token distributions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import re

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Sample Corpus

We'll use a small corpus to demonstrate tokenization approaches.

In [ ]:
corpus = [
    "The cat sat on the mat.",
    "The dog played in the park.",
    "A cat and a dog are friends.",
    "The park is near the house.",
    "She played with the cat and the dog.",
    "Natural language processing is fascinating.",
    "Tokenization is the first step in NLP.",
    "Large language models use subword tokenization.",
    "Embeddings represent words as dense vectors.",
    "Transformers revolutionized natural language processing."
]

print(f"Corpus size: {len(corpus)} sentences")
print(f"Total characters: {sum(len(s) for s in corpus)}")

---
## 1. Character-Level Tokenizer

The simplest approach: every character (including spaces and punctuation) becomes a token.

**Pros:** Very small vocabulary, handles any text, no out-of-vocabulary (OOV) issues.

**Cons:** Very long sequences, harder for models to learn word-level semantics.

In [ ]:
class CharTokenizer:
    """Character-level tokenizer."""
    
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
        self.vocab_size = 0
        # Reserve special tokens
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
    
    def build_vocab(self, texts):
        """Build vocabulary from a list of texts."""
        # Start with special tokens
        self.char_to_id = {self.pad_token: 0, self.unk_token: 1}
        
        # Collect all unique characters
        chars = set()
        for text in texts:
            chars.update(text)
        
        # Assign IDs to each character
        for i, char in enumerate(sorted(chars), start=2):
            self.char_to_id[char] = i
        
        self.id_to_char = {v: k for k, v in self.char_to_id.items()}
        self.vocab_size = len(self.char_to_id)
        return self
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        return [self.char_to_id.get(c, self.char_to_id[self.unk_token]) for c in text]
    
    def decode(self, ids):
        """Convert list of token IDs back to text."""
        return ''.join(self.id_to_char.get(i, self.unk_token) for i in ids 
                       if i != self.char_to_id[self.pad_token])
    
    def tokenize(self, text):
        """Return list of character tokens."""
        return list(text)


# Build and test
char_tok = CharTokenizer().build_vocab(corpus)

print(f"Vocabulary size: {char_tok.vocab_size}")
print(f"\nVocabulary: {dict(list(char_tok.char_to_id.items())[:15])} ...")

test_text = "The cat sat."
encoded = char_tok.encode(test_text)
decoded = char_tok.decode(encoded)

print(f"\nOriginal:  '{test_text}'")
print(f"Tokens:    {char_tok.tokenize(test_text)}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{decoded}'")
print(f"Roundtrip: {test_text == decoded}")

---
## 2. Word-Level Tokenizer

Split text on whitespace and punctuation. Each word gets its own token ID.

**Pros:** Tokens have clear semantic meaning, shorter sequences.

**Cons:** Large vocabulary, can't handle unseen words (OOV problem).

In [ ]:
class WordTokenizer:
    """Word-level tokenizer with simple regex splitting."""
    
    def __init__(self, lower=True):
        self.word_to_id = {}
        self.id_to_word = {}
        self.vocab_size = 0
        self.lower = lower
        self.pad_token = '<PAD>'
        self.unk_token = '<UNK>'
    
    def _split(self, text):
        """Split text into words and punctuation."""
        if self.lower:
            text = text.lower()
        # Split on word boundaries, keeping punctuation as separate tokens
        return re.findall(r"\w+|[^\w\s]", text)
    
    def build_vocab(self, texts, min_freq=1):
        """Build vocabulary from texts, optionally filtering by minimum frequency."""
        self.word_to_id = {self.pad_token: 0, self.unk_token: 1}
        
        # Count word frequencies
        word_counts = Counter()
        for text in texts:
            word_counts.update(self._split(text))
        
        # Add words meeting minimum frequency
        for word, count in sorted(word_counts.items()):
            if count >= min_freq:
                self.word_to_id[word] = len(self.word_to_id)
        
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}
        self.vocab_size = len(self.word_to_id)
        self.word_counts = word_counts
        return self
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        return [self.word_to_id.get(w, self.word_to_id[self.unk_token]) 
                for w in self._split(text)]
    
    def decode(self, ids):
        """Convert token IDs back to text."""
        words = [self.id_to_word.get(i, self.unk_token) for i in ids 
                 if i != self.word_to_id[self.pad_token]]
        return ' '.join(words)
    
    def tokenize(self, text):
        """Return list of word tokens."""
        return self._split(text)


# Build and test
word_tok = WordTokenizer().build_vocab(corpus)

print(f"Vocabulary size: {word_tok.vocab_size}")
print(f"\nSample vocabulary:")
for word, idx in list(word_tok.word_to_id.items())[:15]:
    print(f"  '{word}' -> {idx}")

test_text = "The cat sat on the mat."
encoded = word_tok.encode(test_text)
decoded = word_tok.decode(encoded)

print(f"\nOriginal:  '{test_text}'")
print(f"Tokens:    {word_tok.tokenize(test_text)}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{decoded}'")

---
## 3. Byte-Pair Encoding (BPE) Concept

BPE is the tokenization method used by GPT models. The core idea:

1. Start with a character-level vocabulary
2. Find the most frequent pair of adjacent tokens
3. Merge that pair into a new token
4. Repeat until desired vocabulary size is reached

This creates **subword** tokens - a middle ground between character and word level.

In [ ]:
def get_pair_frequencies(vocab):
    """Count frequency of adjacent symbol pairs in the vocabulary."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """Merge all occurrences of a symbol pair in the vocabulary."""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab


def run_bpe(texts, num_merges=20):
    """Run BPE algorithm for a given number of merges."""
    # Build initial vocabulary: words split into characters, with end-of-word marker
    word_freqs = Counter()
    for text in texts:
        for word in text.lower().split():
            # Clean punctuation for simplicity
            word = re.sub(r'[^\w]', '', word)
            if word:
                word_freqs[word] += 1
    
    # Convert to space-separated characters with end-of-word marker
    vocab = {}
    for word, freq in word_freqs.items():
        vocab[' '.join(list(word)) + ' </w>'] = freq
    
    print(f"Initial vocabulary (character-level):")
    for word, freq in sorted(vocab.items(), key=lambda x: -x[1])[:5]:
        print(f"  {word}: {freq}")
    print()
    
    merges = []
    for i in range(num_merges):
        pairs = get_pair_frequencies(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_pair(best_pair, vocab)
        merges.append(best_pair)
        print(f"Merge {i+1}: {best_pair[0]} + {best_pair[1]} -> {''.join(best_pair)} (freq: {pairs[best_pair]})")
    
    print(f"\nFinal vocabulary after {len(merges)} merges:")
    for word, freq in sorted(vocab.items(), key=lambda x: -x[1])[:8]:
        print(f"  {word}: {freq}")
    
    return vocab, merges


bpe_vocab, bpe_merges = run_bpe(corpus, num_merges=15)

---
## 4. Padding Sequences

Models need fixed-length inputs. We pad shorter sequences and (optionally) truncate longer ones.

In [ ]:
def pad_sequences(sequences, max_len=None, pad_value=0):
    """Pad sequences to uniform length."""
    if max_len is None:
        max_len = max(len(seq) for seq in sequences)
    
    padded = []
    for seq in sequences:
        if len(seq) >= max_len:
            padded.append(seq[:max_len])
        else:
            padded.append(seq + [pad_value] * (max_len - len(seq)))
    return np.array(padded)


# Encode all corpus sentences
encoded_sentences = [word_tok.encode(sent) for sent in corpus]

print("Encoded lengths:", [len(s) for s in encoded_sentences])

# Pad to uniform length
padded = pad_sequences(encoded_sentences)
print(f"\nPadded shape: {padded.shape}")
print(f"\nFirst 3 padded sequences:")
for i in range(3):
    print(f"  {corpus[i]}")
    print(f"  -> {padded[i].tolist()}")

---
## 5. Visualization: Token Frequency Distribution

In [ ]:
# Word frequency distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Most common words
most_common = word_tok.word_counts.most_common(20)
words, counts = zip(*most_common)

axes[0].barh(range(len(words)), counts, color='steelblue')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 Most Frequent Tokens (Word-Level)')

# Character frequency from corpus
char_counts = Counter()
for text in corpus:
    char_counts.update(text.lower())

# Filter to alphabetic characters
alpha_counts = {k: v for k, v in char_counts.items() if k.isalpha()}
sorted_chars = sorted(alpha_counts.items(), key=lambda x: -x[1])[:20]
chars, c_counts = zip(*sorted_chars)

axes[1].bar(chars, c_counts, color='coral')
axes[1].set_xlabel('Character')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Character Frequency Distribution')

plt.tight_layout()
plt.show()

## 6. Vocabulary Coverage vs Corpus Size

How does vocabulary size grow as we add more text? This demonstrates Heaps' Law: vocabulary grows sub-linearly with corpus size.

In [ ]:
# Simulate vocabulary growth by incrementally adding sentences
vocab_sizes_word = []
vocab_sizes_char = []
corpus_sizes = []

seen_words = set()
seen_chars = set()
total_tokens = 0

for sentence in corpus:
    words = re.findall(r"\w+", sentence.lower())
    total_tokens += len(words)
    seen_words.update(words)
    seen_chars.update(sentence.lower())
    
    corpus_sizes.append(total_tokens)
    vocab_sizes_word.append(len(seen_words))
    vocab_sizes_char.append(len(seen_chars))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(corpus_sizes, vocab_sizes_word, 'o-', color='steelblue', linewidth=2, 
        markersize=8, label='Word vocabulary')
ax.plot(corpus_sizes, vocab_sizes_char, 's-', color='coral', linewidth=2, 
        markersize=8, label='Character vocabulary')

ax.set_xlabel('Corpus Size (total tokens)', fontsize=12)
ax.set_ylabel('Vocabulary Size', fontsize=12)
ax.set_title('Vocabulary Coverage vs Corpus Size', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final word vocabulary size: {vocab_sizes_word[-1]}")
print(f"Final character vocabulary size: {vocab_sizes_char[-1]}")
print(f"Total tokens processed: {corpus_sizes[-1]}")

## Key Takeaways

| Method | Vocab Size | Sequence Length | OOV Handling | Used By |
|--------|-----------|----------------|--------------|--------|
| Character-level | Very small (~100) | Very long | No OOV | Some char-CNN models |
| Word-level | Very large (100K+) | Short | OOV problem | Traditional NLP |
| BPE (subword) | Medium (~30-50K) | Medium | Rare OOV | GPT, RoBERTa |
| WordPiece | Medium (~30K) | Medium | Rare OOV | BERT |

Modern LLMs almost exclusively use **subword tokenization** (BPE or variants) because it balances vocabulary size with sequence length and gracefully handles rare/unseen words.